# 02 · Explore the Warehouse

**Warehouse:** `data/warehouse/iiot.duckdb`, a DuckDB star schema built from the Silver and Gold layers.

**Goal of this notebook:** look inside the data model and answer business questions with SQL:

1. What tables and views are in the warehouse, and how do they connect?
2. How did the fleet perform month by month?
3. Which production lines, components and machines cost the most?
4. Do the sensors and error codes give early warning of failures?

**Before you run it:**
- Select the project's `.venv` as the notebook kernel (top right in VS Code).
- Build the data and the warehouse once: `iiot run`, then
  `python -c "from iiot.model.warehouse import build_warehouse; build_warehouse()"`.
- The notebook opens the warehouse **read-only**. Run the last cell (or restart the kernel)
  before rebuilding the warehouse, otherwise the rebuild cannot replace the file.

## 0. Setup

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

from iiot.config import PROJECT_ROOT
from iiot.model.warehouse import connect, table_counts, view_names

con = connect()  # read-only


def q(sql: str) -> pd.DataFrame:
    """Run a SQL query on the warehouse and return a DataFrame."""
    return con.sql(sql).df()


# Chart style: validated colour-blind-safe categorical palette (fixed order), recessive grid.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # blue, orange, aqua, yellow
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.color": "#898781", "ytick.color": "#898781", "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE), "lines.linewidth": 2,
    "legend.frameon": False, "legend.fontsize": 8, "figure.dpi": 110,
})
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
LAKH = 1e5  # costs are in INR; 1 lakh = 100,000
print("Connected to the warehouse (read-only)")

## 1. What is in the warehouse?

A **star schema**: *fact tables* hold the events and measurements (one row per reading,
failure, repair, error or machine-month); *dimension tables* describe them (which machine,
which day, which component). Facts point to dimensions through keys such as `machine_id`
and `date_key`. *Views* (`v_*`) are saved queries that answer common business questions.

In [ ]:
tables = pd.DataFrame(table_counts(con).items(), columns=["table", "rows"])
tables["kind"] = tables["table"].str.split("_").str[0].map({"dim": "dimension", "fact": "fact"})
print("Views:", ", ".join(view_names(con)))
tables

In [ ]:
# The columns of one table (change the name to look at another)
q("DESCRIBE fact_failure")[["column_name", "column_type", "null"]]

In [ ]:
q("SELECT * FROM dim_machine LIMIT 5")

## 2. The star schema in one query

Join a fact table to its dimensions to slice it by anything they describe.
Here: failure cost per plant and component, in lakh INR.

In [ ]:
q("""
    SELECT m.plant_name, f.component, count(*) AS failures,
           round(sum(f.failure_cost) / 1e5, 1) AS failure_cost_lakh
    FROM fact_failure AS f
    JOIN dim_machine  AS m USING (machine_id)
    JOIN dim_date     AS d USING (date_key)
    WHERE d.year = 2015
    GROUP BY m.plant_name, f.component
    ORDER BY m.plant_name, f.component
""").pivot(index="component", columns="plant_name", values="failure_cost_lakh")

## 3. Fleet performance month by month (`v_fleet_monthly`)

In [ ]:
fleet = q("SELECT * FROM v_fleet_monthly")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

axes[0].bar(fleet["year_month"], fleet["failures"], color=PALETTE[0], width=0.6)
axes[0].set_title("Failure events per month")
axes[0].set_ylabel("failure events")
ax2 = axes[0].twinx()
ax2.plot(fleet["year_month"], fleet["availability"] * 100, color=PALETTE[1], marker="o", markersize=3)
ax2.set_ylabel("availability (%)", color=PALETTE[1])
ax2.grid(False)
ax2.spines["right"].set_visible(True)

axes[1].bar(fleet["year_month"], fleet["failure_cost"] / LAKH, color=PALETTE[1], width=0.6,
            label="unplanned failures")
axes[1].bar(fleet["year_month"], fleet["maintenance_cost"] / LAKH, color=PALETTE[0], width=0.6,
            bottom=fleet["failure_cost"] / LAKH, label="planned maintenance")
axes[1].set_title("Cost per month")
axes[1].set_ylabel("lakh INR")
axes[1].set_ylim(0, (fleet["total_cost"] / LAKH).max() * 1.25)
axes[1].legend(ncol=2, loc="upper left")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

total = fleet[["failures", "downtime_h", "failure_cost", "maintenance_cost", "total_cost"]].sum()
print(f"Year total: {total['failures']:.0f} failure events, {total['downtime_h']:,.0f} downtime hours, "
      f"{total['total_cost'] / 1e7:,.1f} crore INR ({total['failure_cost'] / total['total_cost']:.0%} from unplanned failures)")

## 4. Production lines (`v_line_performance`)

In [ ]:
lines = q("SELECT *, availability * 100 AS availability_pct FROM v_line_performance ORDER BY availability_rank")
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(lines["line_id"], lines["cost_per_machine"] / LAKH, color=PALETTE[0], height=0.6)
ax.invert_yaxis()
ax.set_title("Cost per machine by line (best availability at the top)")
ax.set_xlabel("lakh INR per machine")
ax.grid(axis="x", color="#e1e0d9")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()
lines[["availability_rank", "plant_name", "line_id", "machines", "failures", "availability_pct",
       "mtbf_h", "mttr_h", "failures_per_machine", "cost_per_machine"]]

## 5. Components (`v_component_reliability`)

`saving_if_all_prevented` is what the business would save if every failure had been caught
early and replaced as planned maintenance instead. This is the prize for the ML model.

In [ ]:
comps = q("SELECT * FROM v_component_reliability ORDER BY failure_cost DESC")
fig, ax = plt.subplots(figsize=(7, 3))
x = range(len(comps))
ax.bar([i - 0.2 for i in x], comps["failure_cost"] / LAKH, width=0.4, color=PALETTE[1], label="failure cost")
ax.bar([i + 0.2 for i in x], comps["saving_if_all_prevented"] / LAKH, width=0.4, color=PALETTE[2],
       label="saving if all prevented")
ax.set_xticks(list(x), comps["component"])
ax.set_title("Failure cost and potential saving per component")
ax.set_ylabel("lakh INR")
ax.legend()
plt.tight_layout()
plt.show()
comps[["component", "failures", "machines_affected", "unplanned_share",
       "mean_days_between_failures", "failure_cost", "saving_if_all_prevented"]]

## 6. Machines that need attention (`v_machine_health`)

In [ ]:
q("""
    SELECT cost_rank, machine_id, model, age, line_id, failures,
           availability * 100 AS availability_pct,
           errors_last_30d, days_since_last_failure, total_cost
    FROM v_machine_health
    ORDER BY cost_rank
    LIMIT 10
""")

## 7. Sensors before a failure

The daily average of each sensor for the machine with the most failures; dashed lines
mark its failures. Readings with quality `missing` are left out.

In [ ]:
machine = q("SELECT machine_id FROM v_machine_health ORDER BY failures DESC, machine_id LIMIT 1").iloc[0, 0]
daily = q(f"""
    SELECT date_trunc('day', timestamp) AS day, sensor, avg(value) AS value
    FROM fact_sensor_reading
    WHERE machine_id = {machine} AND quality <> 'missing'
    GROUP BY ALL ORDER BY day
""").pivot(index="day", columns="sensor", values="value")
failures = q(f"SELECT DISTINCT timestamp FROM fact_failure WHERE machine_id = {machine}")["timestamp"]

fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
for ax, sensor, colour in zip(axes, daily.columns, PALETTE):
    ax.plot(daily.index, daily[sensor], color=colour, linewidth=1.2)
    for t in failures:
        ax.axvline(t, color="#52514e", linestyle="--", linewidth=0.8)
    ax.set_ylabel(sensor)
axes[0].set_title(f"Machine {machine}: daily sensor averages and failures (dashed)")
plt.tight_layout()
plt.show()

## 8. All example queries (`docs/example_queries.sql`)

Ten business questions, each answered by one SQL query. The file is also tested, so the
queries always match the current schema.

In [ ]:
sql_text = (PROJECT_ROOT / "docs" / "example_queries.sql").read_text(encoding="utf-8")
titles = re.findall(r"^-- (Q\d+\..*)$", sql_text, flags=re.MULTILINE)
statements = con.extract_statements(sql_text)
for title, statement in zip(titles, statements):
    print(title)
    display(con.execute(statement).df())

## 9. Your own query

Change the SQL and run the cell. Tip: start from a view (`v_*`) or join a fact table to its dimensions.

In [ ]:
q("""
    SELECT d.month_name, count(*) AS errors
    FROM fact_error AS e
    JOIN dim_date AS d USING (date_key)
    GROUP BY d.month, d.month_name
    ORDER BY d.month
""")

## 10. Close the connection

Run this before rebuilding the warehouse.

In [ ]:
con.close()
print("Connection closed")